### Dataset and Task Metadata

In [49]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="dementia_prediction",
    dataset_year="2010",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://doi.org/10.17632/tsy6rbc5d4.1",
    download_description="""
We get the data from Mendeley version of another paper (https://www.sciencedirect.com/science/article/pii/S2352914819300917?via%3Dihub).

wget https://data.mendeley.com/public-files/datasets/tsy6rbc5d4/files/b4978a7c-df15-46b2-be69-891d4935b652/file_downloaded && mv file_downloaded oasis_longitudinal_demographics.xlsx && mkdir -p local-data-warehouse/dementia_prediction && mv oasis_longitudinal_demographics.xlsx local-data-warehouse/dementia_prediction/
""",
    # References
    academic_reference_bibtex="""@article{marcus2010open,
  title={Open access series of imaging studies: longitudinal MRI data in nondemented and demented older adults},
  author={Marcus, Daniel S and Fotenos, Anthony F and Csernansky, John G and Morris, John C and Buckner, Randy L},
  journal={Journal of cognitive neuroscience},
  volume={22},
  number={12},
  pages={2677--2684},
  year={2010},
}
""",
    academic_reference_bibtex_key="marcus2010open",
    license="CC BY NC 3.0",
    data_tags=["IID", "WrongDomain"],
    curation_comments="""
We start with the data from Mendeley.

- CDR and Group are both the same target variable. CDR is the Clinical Dementia Rating (0 = no dementia, 0.5 = very mild AD, 1 = mild AD, 2 = moderate AD), which determines the group variable. The cases for converted / changed their dementia rating over time. We make this a task to predict for a patient (at any given time point of a scan) their dementia rating. The rating is ordinal but discrete, so we treat it as a classification problem (not regression). We drop cases with moderate_AD (n=3), as we do not have enough data on this class to include them in our prediction task.
- We drop the group variable and any information about the time of the scan as our goal to predict the rating from eTIV, nWBV, and ASF which are all derived from the MRI scan and independent of time.
- We also drop the constant hand column.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="CDR",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="CDR",
    group_on="Subject ID",
)

## Preprocessing

In [36]:
import pandas as pd

df = pd.read_excel(dataset_mold.path / "oasis_longitudinal_demographics.xlsx")
print("Loaded data shape:", df.shape)

df= df.drop(columns=[
    "MRI ID", "Visit", "MR Delay", # uninformative time information
    "Group", # leak
    "Hand", # constant
])


# 0 = no dementia, 0.5 = very mild AD, 1 = mild AD, 2 = moderate AD)
df["CDR"] = df["CDR"].replace({0.5: "very_mild_AD", 1: "mild_AD", 2: "moderate_AD", 0: "no_dementia"})
df= df[df["CDR"] != "moderate_AD"]
as_cat_type = ["M/F", "CDR", "Subject ID"]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).sort_values(by=["Subject ID"]).reset_index(drop=True)

Loaded data shape: (373, 15)


## Data Checks

In [38]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 370
Columns: 10
Use sampling: False (sample size: 370)
Get row duplicates (staged, merged)...
Using top-9 columns for initial filtering: ['nWBV', 'eTIV', 'ASF', 'Subject ID', 'Age', 'MMSE', 'EDUC', 'SES', 'M/F']
Rows remaining as candidates after top-9 filter: 0 (of 370)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [39]:
# Sample Rows
df_head

,Subject ID,M/F,Age,EDUC,SES,MMSE,CDR,eTIV,nWBV,ASF
0,OAS2_0001,M,88,14,2.0,30.0,no_dementia,2004.479526,0.681062,0.875539
1,OAS2_0001,M,87,14,2.0,27.0,no_dementia,1986.550000,0.696106,0.883440
2,OAS2_0002,M,75,12,NaN,23.0,very_mild_AD,1678.290000,0.736336,1.045710
3,OAS2_0002,M,80,12,NaN,22.0,very_mild_AD,1697.911134,0.701236,1.033623
4,OAS2_0002,M,76,12,NaN,28.0,very_mild_AD,1737.620000,0.713402,1.010000


In [40]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Subject ID,category,0.0,0.00,150.0,"OAS2_0073, OAS2_0048, OAS2_0127, OAS2_0070, OAS2_0027, OAS2_0017, OAS2_0034, OAS2_0036, OAS2_0067, OAS2_0037"
1,M/F,category,0.0,0.00,2.0,"F, M"
2,CDR,category,0.0,0.00,3.0,"no_dementia, very_mild_AD, mild_AD"
3,SES,float64,19.0,5.14,5.0,"2.0, 1.0, 3.0, 4.0, 5.0"
4,MMSE,float64,2.0,0.54,18.0,"30.0, 29.0, 28.0, 27.0, 26.0, 23.0, 25.0, 21.0, 20.0, 22.0"
5,eTIV,float64,0.0,0.00,368.0,"1364.5, 1402.1, 1293.11, 1294.81, 1517.3, 1475.33, 1408.58, 1486.07, 1487.6462, 1482.38"
6,nWBV,float64,0.0,0.00,370.0,"0.8012, 0.6811, 0.6961, 0.7363, 0.7012, 0.7134, 0.7095, 0.7182, 0.7111, 0.7115"
7,ASF,float64,0.0,0.00,368.0,"1.2862, 1.2517, 1.3572, 1.3554, 1.1567, 1.1896, 1.2459, 1.181, 1.1797, 1.1839"
8,Age,int64,0.0,0.00,38.0,"73, 75, 78, 80, 71, 81, 82, 77, 76, 68"
9,EDUC,int64,0.0,0.00,12.0,"12, 16, 18, 14, 13, 15, 20, 11, 8, 17"


In [41]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Age,370.0,76.948649,7.592612,60.000000,97.000000
EDUC,370.0,14.578378,2.871327,6.000000,23.000000
SES,351.0,2.467236,1.133103,1.000000,5.000000
MMSE,368.0,27.399457,3.624124,4.000000,30.000000
eTIV,370.0,1487.716040,176.411126,1105.652499,2004.479526
nWBV,370.0,0.729742,0.037047,0.644399,0.836842
ASF,370.0,1.195839,0.138345,0.875539,1.587298


In [42]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column     rank                            
CDR        1      no_dementia    206  55.68
           2     very_mild_AD    123  33.24
           3          mild_AD     41  11.08
M/F        1                F    211  57.03
           2                M    159  42.97
Subject ID 1        OAS2_0073      5   1.35
           2        OAS2_0048      5   1.35
           3        OAS2_0127      5   1.35
           4        OAS2_0070      5   1.35
           5        OAS2_0027      4   1.08

In [43]:
# Target Distribution
target_df

,count,pct
CDR,,
no_dementia,206,55.68
very_mild_AD,123,33.24
mild_AD,41,11.08


## Task Curation

In [46]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from sklearn.model_selection import StratifiedGroupKFold

splits = {}
for repeat_i in range(20): # more repeats than usual for the dataset size, as count of unique mice is much lower
    splits[repeat_i] = {}

    sklearn_splits = StratifiedGroupKFold(n_splits=3, random_state=42 + repeat_i, shuffle=True).split(
        X=df,
        y=df[task_mold.target_column_name],
        groups=df[task_mold.group_on],
    )
    print_once = False
    for fold_idx, (train_index, test_index) in enumerate(sklearn_splits):
        # Print len, target col count, and group counts
        train_data = df.iloc[train_index]
        test_data = df.iloc[test_index]

        if not print_once:
            print(f"""Train N: {len(train_index)}, Test N: {len(test_index)}
            Target Distribution:
            \tTrain target distribution: {df.iloc[train_index][task_mold.target_column_name].value_counts(normalize=True).to_dict()}
            \tTest target distribution: {df.iloc[test_index][task_mold.target_column_name].value_counts(normalize=True).to_dict()}
            Group Distribution {task_mold.group_on}:
            \tTrain: {len(train_data[task_mold.group_on].unique())} | {len(train_data)}
            \tTest: {len(test_data[task_mold.group_on].unique())} | {len(test_data)}
            """
            )
            print_once = True
        splits[repeat_i][fold_idx] = (train_index.tolist(), test_index.tolist())

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We create stratified grouped 20-repeated 3-fold split. This creates ca. 50 group members (ca. 120-150 samples) per test set.",
    splits=splits
)

Train N: 243, Test N: 127
            Target Distribution:
            	Train target distribution: {'no_dementia': 0.5967078189300411, 'very_mild_AD': 0.3168724279835391, 'mild_AD': 0.08641975308641975}
            	Test target distribution: {'no_dementia': 0.48031496062992124, 'very_mild_AD': 0.36220472440944884, 'mild_AD': 0.15748031496062992}
            Group Distribution Subject ID:
            	Train: 99 | 243
            	Test: 51 | 127
            
Train N: 244, Test N: 126
            Target Distribution:
            	Train target distribution: {'no_dementia': 0.5737704918032787, 'very_mild_AD': 0.3442622950819672, 'mild_AD': 0.08196721311475409}
            	Test target distribution: {'no_dementia': 0.5238095238095238, 'very_mild_AD': 0.30952380952380953, 'mild_AD': 0.16666666666666666}
            Group Distribution Subject ID:
            	Train: 99 | 244
            	Test: 51 | 126
            
Train N: 253, Test N: 117
            Target Distribution:
            	Train t

## Export

In [50]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c9f70-a01f-7409-a13b-6d5c1a832369
1d31edbb3250dfd93e5d04aa494f5c11b1387822e1488b811a9d6f18addb58b2
